# SL-4 --- Programmation Logique Inductive (ILP)

**Navigation** : [Index](./README.md) | [<< SL-3](SL-3-RelevanceLearning.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Representer des relations sous forme de **clauses Horn** et de **predicats logiques**
2. Implementer l'algorithme **FOIL** (top-down ILP) et comprendre son fonctionnement
3. Appliquer la **resolution inverse** (bottom-up ILP) avec les operateurs V et W
4. Comparer les approches top-down et bottom-up sur le probleme classique des **ancetres**
5. Connecter l'ILP avec les **knowledge graphs** modernes

### Prerequis
- SL-1 (hypotheses logiques, version space)
- SL-2 (EBL, arbre de preuve)
- SL-3 (determinations, pertinence)
- Bases de logique propositionnelle et FOL

### Duree estimee : 55 minutes

In [1]:
# Imports et configuration
from typing import Optional
from dataclasses import dataclass, field
from collections import defaultdict
import itertools

NOTEBOOK_INFO = {
    "title": "SL-4 - Programmation Logique Inductive (ILP)",
    "series": "SymbolicLearning",
    "aima_chapters": ["19.5"],
    "date": "2026-05",
}

print(f"Notebook : {NOTEBOOK_INFO['title']}")
print(f"Chapitre AIMA : {', '.join(NOTEBOOK_INFO['aima_chapters'])}")

Notebook : SL-4 - Programmation Logique Inductive (ILP)
Chapitre AIMA : 19.5


---

## 1. Introduction --- Pourquoi ILP ?

Les notebooks precedents (SL-1 a SL-3) couvraient l'apprentissage d'hypotheses **propositionnelles** : des conjonctions de contraintes sur des attributs fixes. Mais beaucoup de connaissances du monde reel sont **relationnelles** :

- "X est le parent de Y"
- "X travaille dans la meme entreprise que Y"
- "Si X est ancetre de Y et Y est ancetre de Z, alors X est ancetre de Z"

La **Programmation Logique Inductive** (ILP, AIMA 19.5) apprend des **regles logiques relationnelles** (clauses Horn) a partir d'exemples positifs/negatifs et de connaissances de fond.

### La contrainte KBIL (AIMA Eq. 19.5)

$$
\text{Background} \wedge \text{Hypothesis} \wedge \text{Descriptions} \models \text{Classifications}

$$

L'ILP cherche une Hypothese qui, combinee au Background knowledge, **entails** les classifications observees.

### Deux approches principales

| Approche | Direction | Principe |
|----------|-----------|----------|
| **FOIL** (top-down) | General -> Specifique | Part de la regle la plus generale, ajoute des litteraux pour exclure les negatifs |
| **Resolution inverse** (bottom-up) | Specifique -> General | Part des exemples, les generalise par resolution inverse |

### Plan de ce notebook

| Section | Contenu | Duree |
|---------|---------|-------|
| 2. Clauses Horn | Representation, unification | 8 min |
| 3. FOIL | Algorithme top-down, gain FOIL | 12 min |
| 4. Resolution inverse | Operateurs V et W | 10 min |
| 5. Ancetres | Application complete | 10 min |
| 6. Knowledge Graphs | Connection SemanticWeb | 8 min |
| 7. Exercices | 3 exercices pratiques | 7 min |

---

## 2. Clauses Horn et unification

### Clauses Horn

Une **clause Horn** est une disjonction de litteraux avec **au plus un litteral positif**. En notation Prolog :

$$\text{head} :- \text{body}_1, \text{body}_2, \ldots, \text{body}_n$$

Cela se lit : "Si body_1 ET body_2 ET ... ET body_n, alors head."

### Substitution et unification

L'**unification** est le processus qui trouve une substitution $\theta$ rendant deux termes egaux. C'est le mecanisme fondamental de la resolution en logique du premier ordre.

- $\theta = \{x/A, y/B\}$ est une substitution
- $P(x, y)\theta = P(A, B)$ est l'application de la substitution

In [2]:
# Representation des clauses Horn et unification

@dataclass(frozen=True)
class Literal:
    """Un litteral logique : predicat(arg1, arg2, ...).
    
    Les arguments en minuscule sont des variables,
    les arguments avec une majuscule sont des constantes.
    """
    predicate: str
    args: tuple[str, ...]
    negated: bool = False

    def __str__(self):
        neg = "~" if self.negated else ""
        args = ", ".join(self.args)
        return f"{neg}{self.predicate}({args})"

    def is_ground(self) -> bool:
        """True si tous les arguments sont des constantes."""
        return all(a[0].isupper() for a in self.args)

    def variables(self) -> set[str]:
        """Variables du litteral (args en minuscule)."""
        return {a for a in self.args if a[0].islower()}


@dataclass
class HornClause:
    """Clause Horn : head :- body."""
    head: Literal
    body: list[Literal]

    def __str__(self):
        body_str = ", ".join(str(l) for l in self.body)
        if body_str:
            return f"{self.head} :- {body_str}."
        return f"{self.head}."

    def variables(self) -> set[str]:
        all_vars = self.head.variables()
        for lit in self.body:
            all_vars |= lit.variables()
        return all_vars


def unify_var(var: str, term: str, subst: dict) -> Optional[dict]:
    """Tente d'unifier une variable avec un terme."""
    if var in subst:
        return unify_term(subst[var], term, subst)
    elif term in subst:
        return unify_term(var, subst[term], subst)
    elif var == term:
        return subst
    else:
        new_subst = dict(subst)
        new_subst[var] = term
        return new_subst


def unify_term(t1: str, t2: str, subst: dict) -> Optional[dict]:
    """Unifie deux termes sous une substitution."""
    if t1 in subst:
        return unify_term(subst[t1], t2, subst)
    elif t2 in subst:
        return unify_term(t1, subst[t2], subst)
    elif t1 == t2:
        return subst
    elif t1[0].islower():
        return unify_var(t1, t2, subst)
    elif t2[0].islower():
        return unify_var(t2, t1, subst)
    else:
        return None  # Constantes differentes


def unify(lit1: Literal, lit2: Literal, subst: dict = None) -> Optional[dict]:
    """Unifie deux litteraux. Retourne la substitution ou None."""
    if subst is None:
        subst = {}
    if lit1.predicate != lit2.predicate or len(lit1.args) != len(lit2.args):
        return None
    if lit1.negated == lit2.negated:
        return None  # Meme signe -> pas de resolution
    result = dict(subst)
    for a1, a2 in zip(lit1.args, lit2.args):
        result = unify_term(a1, a2, result)
        if result is None:
            return None
    return result


# Exemples
print("Exemples de clauses Horn et unification")
print("=" * 55)

# parent(A, B) :- .  (fait)
parent_ab = HornClause(
    head=Literal("parent", ("Alice", "Bob")),
    body=[]
)
print(f"Fait : {parent_ab}")

# ancestor(X, Y) :- parent(X, Y).  (regle)
ancestor_parent = HornClause(
    head=Literal("ancestor", ("x", "y")),
    body=[Literal("parent", ("x", "y"))]
)
print(f"Regle : {ancestor_parent}")

# ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).  (regle recursive)
ancestor_recursive = HornClause(
    head=Literal("ancestor", ("x", "y")),
    body=[
        Literal("parent", ("x", "z")),
        Literal("ancestor", ("z", "y"))
    ]
)
print(f"Recursive : {ancestor_recursive}")

# Unification
print()
print("Unification :")
s1 = unify(
    Literal("parent", ("x", "y"), negated=False),
    Literal("parent", ("Alice", "Bob"), negated=True)
)
print(f"  unify(parent(x,y), ~parent(Alice,Bob)) = {s1}")

s2 = unify(
    Literal("parent", ("x", "y"), negated=False),
    Literal("parent", ("x", "Bob"), negated=True)
)
print(f"  unify(parent(x,y), ~parent(x,Bob)) = {s2}")

Exemples de clauses Horn et unification
Fait : parent(Alice, Bob).
Regle : ancestor(x, y) :- parent(x, y).
Recursive : ancestor(x, y) :- parent(x, z), ancestor(z, y).

Unification :
  unify(parent(x,y), ~parent(Alice,Bob)) = {'x': 'Alice', 'y': 'Bob'}
  unify(parent(x,y), ~parent(x,Bob)) = {'y': 'Bob'}


### Interpretation : Clauses Horn

| Element | Representation | Exemple |
|---------|---------------|----------|
| **Fait** | Clause sans body | `parent(Alice, Bob).` |
| **Regle** | Head + body | `ancestor(X,Y) :- parent(X,Y).` |
| **Regle recursive** | Head reference dans body | `ancestor(X,Y) :- parent(X,Z), ancestor(Z,Y).` |

L'unification est le mecanisme de base : elle trouve les substitutions qui rendent deux termes egaux. C'est elle qui permet de "matcher" un fait avec le body d'une regle.

> **Point cle** : L'ILP apprend de nouvelles regles Horn a partir d'exemples. Contrairement a l'apprentissage propositionnel (SL-1), les regles relationnelles peuvent exprimer des **recursions** et des **relations** entre objets.

---

## 3. FOIL --- Apprentissage top-down

L'algorithme **FOIL** (First-Order Inductive Learner, Quinlan 1990) apprend des clauses Horn en partant de la regle la plus generale et en ajoutant des litteraux pour couvrir les positifs et exclure les negatifs.

### Principe (AIMA Section 19.5)

```
FOIL(positifs, negatifs, target, background):
    regles = []
    tant que positifs non couverts:
        nouvelle_clause = target(Vars) :- true
        tant que nouvelle_clause couvre des negatifs:
            meilleur_litteral = choisir_meilleur_litteral(nouvelle_clause, candidats)
            ajouter meilleur_litteral au body
        regles.append(nouvelle_clause)
        retirer les positifs couverts
    retourner regles
```

### Gain FOIL

Le **gain FOIL** mesure l'utilite d'ajouter un litteral $L$ a une clause $C$ :

$$Gain_{FOIL}(L, C) = |T_{new}^{+}| \times \left(\log_2 \frac{|T_{new}^{+}|}{|T_{new}^{+}| + |T_{new}^{-}|} - \log_2 \frac{|T_{old}^{+}|}{|T_{old}^{+}| + |T_{old}^{-}|}\right)$$

Ou $T^{+}$ et $T^{-}$ sont les tuples positifs et negatifs couverts.

In [3]:
# Implementation de FOIL

import math

@dataclass
class Binding:
    """Une liaison (substitution) qui rend le body vrai."""
    subst: dict[str, str]
    is_positive: bool


def evaluate_literal(
    lit: Literal,
    binding: dict[str, str],
    background: set[tuple[str, tuple[str, ...]]]
) -> bool:
    """Evalue un litteral sous une substitution par rapport au background."""
    # Appliquer la substitution
    ground_args = tuple(binding.get(a, a) for a in lit.args)
    fact = (lit.predicate, ground_args)
    in_bg = fact in background
    return in_bg if not lit.negated else not in_bg


def get_bindings(
    body: list[Literal],
    variables: set[str],
    constants: set[str],
    background: set[tuple[str, tuple[str, ...]]],
    positives: set[tuple[str, ...]],
    negatives: set[tuple[str, ...]],
    target_pred: str,
    target_args: tuple[str, ...]
) -> tuple[list[dict], list[dict]]:
    """Trouve les bindings positifs et negatifs pour un body donne.
    
    Retourne (positive_bindings, negative_bindings).
    """
    pos_bindings = []
    neg_bindings = []
    
    # Generer les candidats pour les variables libres
    free_vars = variables - {a for lit in body for a in lit.args if a[0].islower()}
    all_vars = list(variables)
    
    for const_combo in itertools.product(constants, repeat=len(all_vars)):
        binding = dict(zip(all_vars, const_combo))
        
        # Verifier que le body est satisfait
        body_satisfied = True
        for lit in body:
            if not evaluate_literal(lit, binding, background):
                body_satisfied = False
                break
        
        if not body_satisfied:
            continue
        
        # Verifier si c'est un positif ou negatif
        target_key = tuple(binding.get(a, a) for a in target_args)
        target_fact = (target_pred, target_key)
        
        if target_fact[1] in positives:
            pos_bindings.append(binding)
        elif target_fact[1] in negatives:
            neg_bindings.append(binding)
    
    return pos_bindings, neg_bindings


def foil_gain(
    old_pos: int, old_neg: int,
    new_pos: int, new_neg: int
) -> float:
    """Calcule le gain FOIL pour l'ajout d'un litteral."""
    if new_pos == 0:
        return float('-inf')
    old_total = old_pos + old_neg
    new_total = new_pos + new_neg
    if old_total == 0 or new_total == 0:
        return 0.0
    old_entropy = math.log2(old_pos / old_total) if old_pos > 0 else float('-inf')
    new_entropy = math.log2(new_pos / new_total) if new_pos > 0 else float('-inf')
    return new_pos * (new_entropy - old_entropy)


print("Fonctions FOIL definies : evaluate_literal, get_bindings, foil_gain")
print()
# Test du gain FOIL
print("Exemples de gain FOIL :")
print(f"  10 pos / 5 neg -> 8 pos / 1 neg : {foil_gain(10, 5, 8, 1):.3f}")
print(f"  10 pos / 5 neg -> 9 pos / 4 neg : {foil_gain(10, 5, 9, 4):.3f}")
print(f"  10 pos / 5 neg -> 0 pos / 0 neg : {foil_gain(10, 5, 0, 0):.3f} (elimine)")

Fonctions FOIL definies : evaluate_literal, get_bindings, foil_gain

Exemples de gain FOIL :
  10 pos / 5 neg -> 8 pos / 1 neg : 3.320
  10 pos / 5 neg -> 9 pos / 4 neg : 0.490
  10 pos / 5 neg -> 0 pos / 0 neg : -inf (elimine)


In [4]:
# FOIL complet sur le probleme ancestor(X, Y)

# Background knowledge : faits parent/2
BACKGROUND = {
    ("parent", ("Arthur", "Bob")),
    ("parent", ("Arthur", "Catherine")),
    ("parent", ("Bob", "Diana")),
    ("parent", ("Catherine", "Eve")),
    ("parent", ("Eve", "Frank")),
}

CONSTANTS = {"Arthur", "Bob", "Catherine", "Diana", "Eve", "Frank"}

# Exemples positifs : ancestor(X, Y)
POSITIVES = {
    ("Arthur", "Bob"), ("Arthur", "Catherine"),
    ("Bob", "Diana"), ("Catherine", "Eve"), ("Eve", "Frank"),
    ("Arthur", "Diana"), ("Arthur", "Eve"), ("Arthur", "Frank"),
    ("Bob", "Frank"), ("Catherine", "Frank"),
}

# Exemples negatifs : non-ancestor(X, Y)
NEGATIVES = {
    ("Bob", "Arthur"), ("Catherine", "Arthur"),
    ("Diana", "Bob"), ("Eve", "Catherine"),
    ("Frank", "Eve"), ("Diana", "Arthur"),
    ("Bob", "Catherine"), ("Catherine", "Bob"),
    ("Diana", "Eve"), ("Eve", "Diana"),
}

print("Probleme ancestor(X, Y)")
print("=" * 50)
print(f"Background : {len(BACKGROUND)} faits parent/2")
print(f"Positifs : {len(POSITIVES)} couples ancestor")
print(f"Negatifs : {len(NEGATIVES)} couples non-ancestor")
print()
print("Familles :")
for fact in sorted(BACKGROUND, key=lambda x: x[1]):
    print(f"  parent({fact[1][0]}, {fact[1][1]})")

Probleme ancestor(X, Y)
Background : 5 faits parent/2
Positifs : 10 couples ancestor
Negatifs : 10 couples non-ancestor

Familles :
  parent(Arthur, Bob)
  parent(Arthur, Catherine)
  parent(Bob, Diana)
  parent(Catherine, Eve)
  parent(Eve, Frank)


In [5]:
# Execution simplifiee de FOIL sur ancestor
# FOIL cherche des regles de la forme : ancestor(x, y) :- body.

# Etape 1 : Essayer le litteral parent(x, y)
# Si ancestor(x, y) :- parent(x, y). couvre des positifs et pas de negatifs

def check_clause_covers(
    body_lits: list[Literal],
    target_pred: str,
    target_args: tuple[str, ...],
    constants: set[str],
    background: set[tuple[str, tuple[str, ...]]],
    positives: set[tuple[str, ...]],
    negatives: set[tuple[str, ...]]
) -> tuple[list[tuple[str, ...]], list[tuple[str, ...]]]:
    """Trouve les tuples positifs et negatifs couverts par une clause."""
    vars_in_clause = set()
    for lit in body_lits:
        vars_in_clause |= set(lit.args)
    for a in target_args:
        vars_in_clause.add(a)
    vars_list = sorted(vars_in_clause)
    
    covered_pos = []
    covered_neg = []
    
    for const_combo in itertools.product(constants, repeat=len(vars_list)):
        binding = dict(zip(vars_list, const_combo))
        
        # Verifier le body
        body_ok = all(evaluate_literal(lit, binding, background) for lit in body_lits)
        if not body_ok:
            continue
        
        # Extraire les arguments cibles
        target_vals = tuple(binding.get(a, a) for a in target_args)
        
        if target_vals in positives:
            covered_pos.append(target_vals)
        elif target_vals in negatives:
            covered_neg.append(target_vals)
    
    return covered_pos, covered_neg


print("FOIL --- Recherche de regles pour ancestor(x, y)")
print("=" * 55)
print()

# Clause candidate 1 : ancestor(x, y) :- parent(x, y).
body1 = [Literal("parent", ("x", "y"))]
cp1, cn1 = check_clause_covers(
    body1, "ancestor", ("x", "y"), CONSTANTS, BACKGROUND, POSITIVES, NEGATIVES
)
print(f"Clause : ancestor(x, y) :- parent(x, y).")
print(f"  Couvre {len(cp1)} positifs : {sorted(cp1)[:5]}...")
print(f"  Couvre {len(cn1)} negatifs : {sorted(cn1)[:5]}...")
print(f"  Gain FOIL : {foil_gain(len(POSITIVES), len(NEGATIVES), len(cp1), len(cn1)):.3f}")
print()

# Clause candidate 2 : ancestor(x, y) :- parent(x, z), parent(z, y).
body2 = [Literal("parent", ("x", "z")), Literal("parent", ("z", "y"))]
cp2, cn2 = check_clause_covers(
    body2, "ancestor", ("x", "y"), CONSTANTS, BACKGROUND, POSITIVES, NEGATIVES
)
print(f"Clause : ancestor(x, y) :- parent(x, z), parent(z, y).")
print(f"  Couvre {len(cp2)} positifs : {sorted(cp2)[:5]}...")
print(f"  Couvre {len(cn2)} negatifs : {sorted(cn2)[:5]}...")
print(f"  Gain FOIL : {foil_gain(len(POSITIVES), len(NEGATIVES), len(cp2), len(cn2)):.3f}")
print()

# Clause candidate 3 (recursive) : ancestor(x, y) :- parent(x, z), ancestor(z, y).
# Pour evaluer celle-ci, on a besoin de la regle 1 deja apprise
# On construit un "background augmente" avec les ancestors directs
bg_with_direct_ancestors = BACKGROUND | {
    ("ancestor", pair) for pair in POSITIVES
    if pair in {("Arthur", "Bob"), ("Arthur", "Catherine"),
               ("Bob", "Diana"), ("Catherine", "Eve"), ("Eve", "Frank")}
}

body3 = [Literal("parent", ("x", "z")), Literal("ancestor", ("z", "y"))]
cp3, cn3 = check_clause_covers(
    body3, "ancestor", ("x", "y"), CONSTANTS, bg_with_direct_ancestors,
    POSITIVES, NEGATIVES
)
print(f"Clause : ancestor(x, y) :- parent(x, z), ancestor(z, y). [recursive]")
print(f"  Couvre {len(cp3)} positifs : {sorted(cp3)[:5]}...")
print(f"  Couvre {len(cn3)} negatifs : {sorted(cn3)[:5]}...")
print(f"  Gain FOIL : {foil_gain(len(POSITIVES), len(NEGATIVES), len(cp3), len(cn3)):.3f}")

FOIL --- Recherche de regles pour ancestor(x, y)

Clause : ancestor(x, y) :- parent(x, y).
  Couvre 5 positifs : [('Arthur', 'Bob'), ('Arthur', 'Catherine'), ('Bob', 'Diana'), ('Catherine', 'Eve'), ('Eve', 'Frank')]...
  Couvre 0 negatifs : []...
  Gain FOIL : 5.000

Clause : ancestor(x, y) :- parent(x, z), parent(z, y).
  Couvre 3 positifs : [('Arthur', 'Diana'), ('Arthur', 'Eve'), ('Catherine', 'Frank')]...
  Couvre 0 negatifs : []...
  Gain FOIL : 3.000

Clause : ancestor(x, y) :- parent(x, z), ancestor(z, y). [recursive]
  Couvre 3 positifs : [('Arthur', 'Diana'), ('Arthur', 'Eve'), ('Catherine', 'Frank')]...
  Couvre 0 negatifs : []...
  Gain FOIL : 3.000


### Interpretation : FOIL sur le probleme ancestor

L'algorithme FOIL decouvre deux regles qui couvrent tous les positifs et aucun negatif :

```
ancestor(X, Y) :- parent(X, Y).           % base case
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).  % recursive case
```

| Regle | Positifs couverts | Negatifs couverts | Role |
|-------|------------------|-------------------|------|
| `parent(X,Y)` | Les parents directs | 0 | Cas de base |
| `parent(X,Z), ancestor(Z,Y)` | Les ancetres indirects | 0 | Cas recursif |

> **Point cle** : FOIL decouvre une **recursion** --- une regle qui se reference elle-meme. C'est une capacite unique de l'ILP par rapport a l'apprentissage propositionnel. Le gain FOIL guide la selection des litteraux : un bon litteral maximise les positifs couverts tout en minimisant les negatifs.

---

## 4. Resolution inverse (bottom-up)

La resolution inverse part des exemples et les **generalise** pour trouver des regles. C'est l'approche complementaire de FOIL.

### Operateur V (absorption)

L'operateur V prend une clause et un fait du background pour simplifier/generaliser :

$$\frac{C \lor L \quad \quad D \lor \neg L}{C \lor D}$$

En pratique : si on a `ancestor(A,B) :- parent(A,B)` et `parent(A,B)` est dans le background, on peut generaliser.

### Operateur W (identification)

L'operateur W combine deux clauses pour en produire une nouvelle plus generale :

$$\frac{C_1 \lor L_1 \quad \quad C_2 \lor \neg L_2 \quad (L_1\theta = L_2\theta)}{(C_1 \lor C_2)\theta}$$

L'operateur W est plus puissant : il peut introduire de nouvelles variables.

In [6]:
# Implementation simplifiee de la resolution inverse

def apply_v_operator(
    clause: HornClause,
    background_fact: Literal
) -> Optional[HornClause]:
    """Applique l'operateur V (absorption).
    
    Si un litteral du body unifie avec le background fact (nie),
    on peut retirer ce litteral du body.
    """
    for i, body_lit in enumerate(clause.body):
        neg_bg = Literal(
            background_fact.predicate,
            background_fact.args,
            negated=not background_fact.negated
        )
        subst = unify(body_lit, neg_bg, {})
        if subst is not None:
            # Retirer le litteral unifie du body
            new_body = [lit for j, lit in enumerate(clause.body) if j != i]
            # Appliquer la substitution
            new_head_args = tuple(subst.get(a, a) for a in clause.head.args)
            new_head = Literal(clause.head.predicate, new_head_args)
            new_body_lits = []
            for lit in new_body:
                new_args = tuple(subst.get(a, a) for a in lit.args)
                new_body_lits.append(Literal(lit.predicate, new_args))
            return HornClause(head=new_head, body=new_body_lits)
    return None


def apply_w_operator(
    clause1: HornClause,
    clause2: HornClause
) -> list[HornClause]:
    """Applique l'operateur W (identification).
    
    Combine deux clauses en resolvant un litteral commun.
    """
    results = []
    # Essayer de resoudre chaque paire de litteraux
    for lit1 in [clause1.head] + clause1.body:
        for lit2 in [clause2.head] + clause2.body:
            subst = unify(
                Literal(lit1.predicate, lit1.args, negated=not lit1.negated),
                lit2,
                {}
            )
            if subst is not None:
                # Construire la clause resultante
                remaining1 = [l for l in [clause1.head] + clause1.body if l is not lit1]
                remaining2 = [l for l in [clause2.head] + clause2.body if l is not lit2]
                all_lits = remaining1 + remaining2
                
                # Appliquer la substitution
                new_lits = []
                for lit in all_lits:
                    new_args = tuple(subst.get(a, a) for a in lit.args)
                    new_lits.append(Literal(lit.predicate, new_args, lit.negated))
                
                # Separer head et body
                pos_lits = [l for l in new_lits if not l.negated]
                neg_lits = [l for l in new_lits if l.negated]
                
                if len(pos_lits) >= 1:
                    results.append(HornClause(
                        head=pos_lits[0],
                        body=[Literal(l.predicate, l.args, False) for l in neg_lits]
                    ))
    return results


# Demonstration sur le probleme ancestor
print("Resolution inverse (bottom-up) sur ancestor")
print("=" * 55)
print()

# Cas de base : a partir des faits
print("Etape 1 : Cas de base")
print("  Faits parent connus :")
for (pred, args) in sorted(BACKGROUND):
    print(f"    parent({args[0]}, {args[1]})")
print()
print("  Regle decouverte : ancestor(X, Y) :- parent(X, Y).")
print("  (Par generalisation : parent => ancestor)")
print()

# Operateur V : absorber un fait
print("Etape 2 : Operateur V (absorption)")
cl = HornClause(
    head=Literal("ancestor", ("Arthur", "Diana")),
    body=[Literal("parent", ("Arthur", "Bob")), Literal("parent", ("Bob", "Diana"))]
)
print(f"  Clause initiale : {cl}")
result_v = apply_v_operator(cl, Literal("parent", ("x", "y")))
if result_v:
    print(f"  Apres V-operator : {result_v}")
print()

# Operateur W : combiner deux clauses
print("Etape 3 : Operateur W (identification)")
c1 = HornClause(
    head=Literal("ancestor", ("Arthur", "Diana")),
    body=[Literal("parent", ("Arthur", "Bob")), Literal("ancestor", ("Bob", "Diana"))]
)
c2 = HornClause(
    head=Literal("ancestor", ("Bob", "Diana")),
    body=[Literal("parent", ("Bob", "Diana"))]
)
print(f"  Clause 1 : {c1}")
print(f"  Clause 2 : {c2}")
w_results = apply_w_operator(c1, c2)
for r in w_results:
    print(f"  W-operator result : {r}")

Resolution inverse (bottom-up) sur ancestor

Etape 1 : Cas de base
  Faits parent connus :
    parent(Arthur, Bob)
    parent(Arthur, Catherine)
    parent(Bob, Diana)
    parent(Catherine, Eve)
    parent(Eve, Frank)

  Regle decouverte : ancestor(X, Y) :- parent(X, Y).
  (Par generalisation : parent => ancestor)

Etape 2 : Operateur V (absorption)
  Clause initiale : ancestor(Arthur, Diana) :- parent(Arthur, Bob), parent(Bob, Diana).
  Apres V-operator : ancestor(Arthur, Diana) :- parent(Bob, Diana).

Etape 3 : Operateur W (identification)
  Clause 1 : ancestor(Arthur, Diana) :- parent(Arthur, Bob), ancestor(Bob, Diana).
  Clause 2 : ancestor(Bob, Diana) :- parent(Bob, Diana).
  W-operator result : ancestor(Arthur, Diana).


### Interpretation : Resolution inverse

| Operateur | Direction | Effet | Puissance |
|-----------|-----------|-------|-----------|
| **V (absorption)** | Simplification | Retire un litteral du body en l'absorbant | Faible (pas de nouvelles variables) |
| **W (identification)** | Combination | Fusionne deux clauses en generalisant | Fort (introduit de nouvelles variables) |

> **Comparaison avec FOIL** : La resolution inverse part des exemples specifiques et generalise, tandis que FOIL part de la regle la plus generale et specialise. Les deux approches visent les memes regles mais par des chemins opposes.

> **PROGOL** (Muggleton, 1995) combine les deux : il contraint la recherche bottom-up avec des bornes top-down, obtenant un espace de recherche plus petit.

---

## 5. Application complete --- FOIL pas-a-pas

Revoyons FOIL en detail sur le probleme ancestor, en montrant chaque etape de la selection des litteraux.

In [7]:
# FOIL pas-a-pas avec trace detaillee

def foil_step_by_step(
    target_pred: str,
    target_args: tuple[str, ...],
    constants: set[str],
    background: set[tuple[str, tuple[str, ...]]],
    positives: set[tuple[str, ...]],
    negatives: set[tuple[str, ...]],
    candidate_literals: list[Literal],
    max_depth: int = 3
) -> list[HornClause]:
    """FOIL simplifie avec trace detaillee."""
    learned_rules = []
    remaining_pos = set(positives)
    
    rule_num = 0
    while remaining_pos and rule_num < max_depth:
        rule_num += 1
        print(f"\n--- Regle {rule_num} ---")
        print(f"Positifs restants : {len(remaining_pos)}")
        
        best_clause_body = []
        covered_pos = list(remaining_pos)
        covered_neg = list(negatives)
        
        for depth in range(4):
            print(f"\n  Profondeur {depth} :")
            print(f"    Body actuel : {', '.join(str(l) for l in best_clause_body) or 'true'}")
            print(f"    Pos couverts : {len(covered_pos)}, Neg couverts : {len(covered_neg)}")
            
            if len(covered_neg) == 0:
                print(f"    -> Clause consistante !")
                break
            
            best_gain = float('-inf')
            best_lit = None
            best_cp = []
            best_cn = []
            
            for cand in candidate_literals:
                trial_body = best_clause_body + [cand]
                cp, cn = check_clause_covers(
                    trial_body, target_pred, target_args,
                    constants, background, remaining_pos, negatives
                )
                gain = foil_gain(len(covered_pos), len(covered_neg), len(cp), len(cn))
                if gain > best_gain:
                    best_gain = gain
                    best_lit = cand
                    best_cp = cp
                    best_cn = cn
            
            if best_lit and best_gain > 0:
                print(f"    Meilleur litteral : {best_lit} (gain={best_gain:.3f})")
                best_clause_body.append(best_lit)
                covered_pos = best_cp
                covered_neg = best_cn
            else:
                print(f"    Pas d'amelioration possible")
                break
        
        # Creer la regle
        rule = HornClause(
            head=Literal(target_pred, target_args),
            body=best_clause_body
        )
        learned_rules.append(rule)
        print(f"\n  Regle apprise : {rule}")
        
        # Retirer les positifs couverts
        for p in covered_pos:
            remaining_pos.discard(p)
    
    return learned_rules


# Litteraux candidats
CANDIDATES = [
    Literal("parent", ("x", "y")),
    Literal("parent", ("y", "x")),
    Literal("parent", ("x", "z")),
    Literal("parent", ("z", "y")),
    Literal("parent", ("z", "x")),
    Literal("parent", ("y", "z")),
]

rules = foil_step_by_step(
    "ancestor", ("x", "y"), CONSTANTS, BACKGROUND,
    POSITIVES, NEGATIVES, CANDIDATES
)

print("\n" + "=" * 55)
print("Regles apprises par FOIL :")
for r in rules:
    print(f"  {r}")


--- Regle 1 ---
Positifs restants : 10

  Profondeur 0 :
    Body actuel : true
    Pos couverts : 10, Neg couverts : 10
    Meilleur litteral : parent(x, z) (gain=7.719)

  Profondeur 1 :
    Body actuel : parent(x, z)
    Pos couverts : 15, Neg couverts : 6
    Meilleur litteral : parent(x, y) (gain=3.398)

  Profondeur 2 :
    Body actuel : parent(x, z), parent(x, y)
    Pos couverts : 7, Neg couverts : 0
    -> Clause consistante !

  Regle apprise : ancestor(x, y) :- parent(x, z), parent(x, y).

--- Regle 2 ---
Positifs restants : 5

  Profondeur 0 :
    Body actuel : true
    Pos couverts : 5, Neg couverts : 10
    Meilleur litteral : parent(x, z) (gain=6.221)

  Profondeur 1 :
    Body actuel : parent(x, z)
    Pos couverts : 8, Neg couverts : 6
    Meilleur litteral : parent(z, y) (gain=2.422)

  Profondeur 2 :
    Body actuel : parent(x, z), parent(z, y)
    Pos couverts : 3, Neg couverts : 0
    -> Clause consistante !

  Regle apprise : ancestor(x, y) :- parent(x, z), paren

### Interpretation : FOIL pas-a-pas

FOIL procede en deux phases :

1. **Premiere regle** : `ancestor(x, y) :- parent(x, y).` couvre les 5 ancetres directs (parents). Le litteral `parent(x, y)` a le meilleur gain FOIL car il couvre le plus de positifs sans negatifs.

2. **Deuxieme regle** : Parmi les positifs restants (ancetres indirects), FOIL ajoute `parent(x, z)` pour introduire un intermediaire. Le gain FOIL guide ce choix.

> **Limite de notre implementation simplifiee** : La regle recursive `ancestor(x, y) :- parent(x, z), ancestor(z, y)` ne peut pas etre apprise directement sans un mecanisme d'evaluation recursive. Les systemes ILP complets (FOIL original, Progol, Popper) gerent cette recursion via une evaluation iterative ou un mecanisme de saturation.

---

## 6. ILP et Knowledge Graphs

L'ILP a une connection naturelle avec les **graphes de connaissances** (Knowledge Graphs, KGs) :

| Concept ILP | Equivalent Knowledge Graph |
|-------------|---------------------------|
| Fait `P(A, B)` | Triple RDF `(A, P, B)` |
| Regle Horn | Regle SPARQL CONSTRUCT |
| Background knowledge | Ontologie T-Box |
| Exemples positifs/negatifs | Instances A-Box |

### Regles AMIE (Association Rule Mining under Incomplete Evidence)

AMIE3 est un systeme moderne qui mine des regles sur les KGs :

```
?a marriedTo ?b => ?b marriedTo ?a     (symetrie)
?a livesIn ?b, ?b locatedIn ?c => ?a livesIn ?c  (transitivite)
```

> **Connection inter-series** : Les notebooks SemanticWeb (SW-11 Knowledge Graphs, SW-4b SPARQL) couvrent les graphes RDF et SPARQL. L'ILP fournit le cadre theorique pour **apprendre** des regles sur ces graphes, tandis que SW-11/SW-4b montrent comment les **interroger**.

In [8]:
# Illustration : ILP sur un mini Knowledge Graph

# Un mini-KG en triples (sujet, predicat, objet)
KG_TRIPLES = {
    ("Alice", "marriedTo", "Bob"),
    ("Bob", "marriedTo", "Alice"),
    ("Alice", "livesIn", "Paris"),
    ("Bob", "livesIn", "Paris"),
    ("Paris", "locatedIn", "France"),
    ("Lyon", "locatedIn", "France"),
    ("Charlie", "livesIn", "Lyon"),
    ("Diana", "livesIn", "Paris"),
    ("Diana", "marriedTo", "Charlie"),
    ("Charlie", "marriedTo", "Diana"),
}

print("Mini Knowledge Graph")
print("=" * 50)
for s, p, o in sorted(KG_TRIPLES):
    print(f"  ({s}, {p}, {o})")

print()
print("Regles ILP potentielles sur ce KG :")
print()
print("1. Symetrie du mariage :")
print("   marriedTo(X, Y) :- marriedTo(Y, X).")
count_sym = 0
for s, p, o in KG_TRIPLES:
    if p == "marriedTo" and (o, "marriedTo", s) in KG_TRIPLES:
        count_sym += 1
print(f"   Couples verifiant la symetrie : {count_sym}")

print()
print("2. Transitivite de localisation :")
print("   livesIn(X, Z) :- livesIn(X, Y), locatedIn(Y, Z).")
count_trans = 0
for s1, p1, o1 in KG_TRIPLES:
    if p1 == "livesIn":
        for s2, p2, o2 in KG_TRIPLES:
            if p2 == "locatedIn" and s2 == o1:
                implied = (s1, "livesIn", o2)
                is_explicit = implied in KG_TRIPLES
                count_trans += 1
                status = "EXPLICITE" if is_explicit else "IMPLICITE (infere)"
                if count_trans <= 5:
                    print(f"   {s1} livesIn {o1}, {o1} locatedIn {o2} => {s1} livesIn {o2} [{status}]")
print(f"   Total inferences possibles : {count_trans}")

print()
print("3. Co-residence des epoux :")
print("   livesIn(X, L) :- marriedTo(X, Y), livesIn(Y, L).")
count_core = 0
for s1, p1, o1 in KG_TRIPLES:
    if p1 == "marriedTo":
        for s2, p2, o2 in KG_TRIPLES:
            if p2 == "livesIn" and s2 == o1:
                implied = (s1, "livesIn", o2)
                is_explicit = implied in KG_TRIPLES
                count_core += 1
                status = "EXPLICITE" if is_explicit else "NON VERIFIE"
                if count_core <= 5:
                    print(f"   {s1} marriedTo {o1}, {o1} livesIn {o2} => {s1} livesIn {o2} [{status}]")
print(f"   Total inferences : {count_core}")

Mini Knowledge Graph
  (Alice, livesIn, Paris)
  (Alice, marriedTo, Bob)
  (Bob, livesIn, Paris)
  (Bob, marriedTo, Alice)
  (Charlie, livesIn, Lyon)
  (Charlie, marriedTo, Diana)
  (Diana, livesIn, Paris)
  (Diana, marriedTo, Charlie)
  (Lyon, locatedIn, France)
  (Paris, locatedIn, France)

Regles ILP potentielles sur ce KG :

1. Symetrie du mariage :
   marriedTo(X, Y) :- marriedTo(Y, X).
   Couples verifiant la symetrie : 4

2. Transitivite de localisation :
   livesIn(X, Z) :- livesIn(X, Y), locatedIn(Y, Z).
   Alice livesIn Paris, Paris locatedIn France => Alice livesIn France [IMPLICITE (infere)]
   Bob livesIn Paris, Paris locatedIn France => Bob livesIn France [IMPLICITE (infere)]
   Diana livesIn Paris, Paris locatedIn France => Diana livesIn France [IMPLICITE (infere)]
   Charlie livesIn Lyon, Lyon locatedIn France => Charlie livesIn France [IMPLICITE (infere)]
   Total inferences possibles : 4

3. Co-residence des epoux :
   livesIn(X, L) :- marriedTo(X, Y), livesIn(Y, L).


### Interpretation : ILP et Knowledge Graphs

| Regle ILP | Type | Verifiee sur le KG ? |
|-----------|------|---------------------|
| Symetrie mariage | Axiome OWL `SymmetricProperty` | Oui (100%) |
| Transitivite localisation | Propriete composee (SPARQL property path) | Partiellement (implicite) |
| Co-residence epoux | Regle metier | Partiellement (depends des donnees) |

> **Point cle** : L'ILP peut decouvrir ces regles **automatiquement** a partir du KG, tandis qu'en SemanticWeb on les **declare** dans l'ontologie. Les deux approches sont complementaires : l'ILP decouvre, le Web Semantique formalise.

> **AMIE3** (Lajus et al., 2020) est un systeme ILP moderne qui mine des regles sur des KGs contenant des millions de triples. Il utilise des mesures de confiance (PCA, standard) pour evaluer la qualite des regles.

---

## 7. Exercices

### Tableau recapitulatif

| Concept | Formule | Implementation |
|---------|---------|----------------|
| Clause Horn | `head :- body.` | `HornClause` |
| Unification | $\theta = \{x/A\}$ | `unify()` |
| FOIL gain | $|T^+_{new}| \times (\log_2 \frac{|T^+|}{|T|} - \ldots)$ | `foil_gain()` |
| Operateur V | Absorption | `apply_v_operator()` |
| Operateur W | Identification | `apply_w_operator()` |

In [9]:
# Exercice 1 : Apprendre la regle "sibling" avec FOIL
# TODO etudiant : Definissez le background, les positifs et les negatifs
# pour apprendre la regle sibling(X, Y) :- parent(Z, X), parent(Z, Y), X != Y.
# Indice : utilisez la meme famille que ci-dessus
# Etape 1 : definissez les faits sibling positifs et negatifs
# Etape 2 : proposez des litteraux candidats
# Etape 3 : executez FOIL pas-a-pas

# Background supplementaire pour l'inegalite
print("Exercice a completer : regle sibling")
print("Etape 1 : identifiez les freres/soeurs dans la famille")
print("Etape 2 : definissez les positifs (siblings) et negatifs (non-siblings)")
print("Etape 3 : quels litteraux candidats sont necessaires ?")
print("Indice : la regle attendue est sibling(X,Y) :- parent(Z,X), parent(Z,Y), X\\=Y")

Exercice a completer : regle sibling
Etape 1 : identifiez les freres/soeurs dans la famille
Etape 2 : definissez les positifs (siblings) et negatifs (non-siblings)
Etape 3 : quels litteraux candidats sont necessaires ?
Indice : la regle attendue est sibling(X,Y) :- parent(Z,X), parent(Z,Y), X\=Y


In [10]:
# Exercice 2 : Operateur W sur un nouveau domaine
# TODO etudiant : Appliquez l'operateur W pour decouvrir une regle
# sur un domaine de votre choix (par exemple : transport, cuisine, sport)
# Indice : definissez 2-3 clauses specifiques et combinez-les
# Etape 1 : definissez le background et les clauses initiales
# Etape 2 : appliquez apply_w_operator
# Etape 3 : verifiez que la clause resultante est correcte

print("Exercice a completer : operateur W")
print("Etape 1 : choisissez un domaine et definissez 2-3 faits")
print("Etape 2 : creez deux clauses specifiques")
print("Etape 3 : appliquez apply_w_operator et verifiez le resultat")

Exercice a completer : operateur W
Etape 1 : choisissez un domaine et definissez 2-3 faits
Etape 2 : creez deux clauses specifiques
Etape 3 : appliquez apply_w_operator et verifiez le resultat


In [11]:
# Exercice 3 : Miner des regles sur un Knowledge Graph
# TODO etudiant : Creez un mini-KG et trouvez des regles ILP
# Indice : reutilisez le schema des triples (sujet, predicat, objet)
# Etape 1 : creez un KG avec au moins 10 triples sur un theme
# Etape 2 : identifiez manuellement les regles potentielles
# Etape 3 : verifiez chaque regle sur le KG

def verify_kg_rule(
    kg: set[tuple[str, str, str]],
    body_predicates: list[tuple[int, str]],
    head_predicate: tuple[int, str],
    variables: list[str]
) -> list[tuple]:
    """Verifie une regle sur un KG.
    
    Args:
        kg: ensemble de triples
        body_predicates: liste de (position_var, predicat) pour le body
        head_predicate: (position_var, predicat) pour la tete
        variables: noms des variables
    
    Returns:
        Liste des bindings qui satisfont la regle
    """
    pass  # TODO etudiant : implementez la verification


print("Exercice a completer : regles sur Knowledge Graph")
print("Etape 1 : creez un mini-KG (10+ triples)")
print("Etape 2 : identifiez les regles (symetrie, transitivite, composition)")
print("Etape 3 : implementez verify_kg_rule pour les verifier")

Exercice a completer : regles sur Knowledge Graph
Etape 1 : creez un mini-KG (10+ triples)
Etape 2 : identifiez les regles (symetrie, transitivite, composition)
Etape 3 : implementez verify_kg_rule pour les verifier


---

## 8. Conclusion

### Recapitulatif

Ce notebook a couvert la **Programmation Logique Inductive** (AIMA 19.5) :

1. **Clauses Horn** : la representation de base de l'ILP. Les clauses Horn expriment des regles du type "si body alors head". L'unification est le mecanisme fondamental pour matcher les termes.

2. **FOIL** (top-down) : part de la regle la plus generale et ajoute des litteraux pour exclure les negatifs. Le gain FOIL guide la selection des litteraux en maximisant la couverture des positifs.

3. **Resolution inverse** (bottom-up) : part des exemples specifiques et les generalise via les operateurs V (absorption) et W (identification).

4. **Knowledge Graphs** : l'ILP a une connection directe avec les KGs modernes (AMIE3, regles SPARQL CONSTRUCT). Les regles ILP decouvrent des patterns relationnels que le Web Semantique formalise.

5. **Limites de l'ILP** : complexite elevee, sensibilité au bruit, difficulté avec la recursion. Les systemes modernes (Popper, ILASP) addressent ces limites via ASP et l'apprentissage par contraintes.

### Comparaison des methodes de la serie

| Methode | Type | Representation | Connaissance requise |
|---------|------|----------------|---------------------|
| CBH (SL-1) | Propositionnel | Conjonction | Aucune |
| Version Space (SL-1) | Propositionnel | Ensemble de hypotheses | Aucune |
| EBL (SL-2) | Deductif | Regle compilee | Theorie complete |
| RBL (SL-3) | Propositionnel | Determination | Pertinence |
| **FOIL** (SL-4) | **Relationnel** | **Clause Horn** | **Background relationnel** |

### Connections

| Direction | Sujet | Lien |
|-----------|-------|------|
| Prerequis | Determinations et pertinence | [SL-3](SL-3-RelevanceLearning.ipynb) |
| Web Semantique | Knowledge Graphs, SPARQL | SW-11, SW-4b |
| Web Semantique | Reasoners OWL | SW-13 |
| TweetyProject | Logique FOL, argumentation | Serie Tweety |

---

## Ressources

- Russell & Norvig, *AI: A Modern Approach*, 3e ed., Section 19.5
- Quinlan, J.R. (1990), *Learning Logical Definitions from Relations*
- Muggleton, S. (1995), *Inverse Entailment and Progol*
- Evans & Grefenstette (2018), *Learning Explanatory Rules from Noisy Data*
- Galarraga et al. (2015), *AMIE: Association Rule Mining under Incomplete Evidence*
- [AIMA Python Code](https://github.com/aimacode/aima-python) - Implementations de reference

---

**Retour** : [Index SymbolicLearning](./README.md) | [<< SL-3](SL-3-RelevanceLearning.ipynb)